In [1]:
from dotenv import load_dotenv

load_dotenv()


from langchain.chat_models import init_chat_model


model = init_chat_model("meta-llama/llama-4-scout-17b-16e-instruct", model_provider="groq")

# 2.4 Wedding Planner
In this lab you will build a multi-agent wedding planner.

> Note: This lab has been updated since filming to make the agent performance more robust to errors and to limit run time. In particular: 1) added error handling for MCP failures, 2) limited the number of searches. It's worth noting that, where possible, tool errors should be returned to the agent rather than failing so that the agent can adjust its invocation and try again.

## Setup Tools


In [2]:
import asyncio
import json as json_mod

from langchain_mcp_adapters.client import MultiServerMCPClient
from mcp.shared.exceptions import McpError
from mcp.types import CallToolResult, TextContent
from langchain_core.tools import StructuredTool
from pydantic import create_model, Field
from typing import Optional

RETRYABLE_MCP_CODES = {-32603}

class GroqFixInterceptor:
    """Retry transient MCP failures and fix Groq param serialization."""

    def __init__(self, max_retries: int = 3):
        self.max_retries = max_retries

    def _fix_params(self, request):
        if request.args:
            fixed = {}
            for key, value in request.args.items():
                if isinstance(value, str):
                    try:
                        parsed = json_mod.loads(value)
                        if isinstance(parsed, (dict, list)):
                            fixed[key] = parsed
                            continue
                    except (json_mod.JSONDecodeError, TypeError):
                        pass
                fixed[key] = value
            request.args = fixed

    async def __call__(self, request, handler):
        self._fix_params(request)
        last_error = None
        for attempt in range(self.max_retries):
            try:
                return await handler(request)
            except McpError as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(code {exc.error.code}, attempt {attempt+1}/{self.max_retries}): {exc}")
                if exc.error.code not in RETRYABLE_MCP_CODES:
                    return CallToolResult(
                        content=[TextContent(type="text", text=f"Tool call failed (non-retryable): {exc}")],
                        isError=False,
                    )
            except Exception as exc:
                last_error = exc
                print(f"[MCP interceptor] {type(exc).__name__} on {request.name} "
                      f"(attempt {attempt+1}/{self.max_retries}): {exc}")
            if attempt < self.max_retries - 1:
                await asyncio.sleep(2 ** attempt)
        print(f"[MCP interceptor] all {self.max_retries} retries exhausted for {request.name}")
        return CallToolResult(
            content=[TextContent(type="text", text=f"Tool call failed after {self.max_retries} attempts: {last_error}")],
            isError=False,
        )

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    },
    tool_interceptors=[GroqFixInterceptor()],
)

mcp_tools = await client.get_tools()

# --- Groq-compatible tool wrapping ---
# Groq validates tool params BEFORE our code runs. It struggles with:
# 1. object-typed params (sends them as JSON strings)
# 2. integer params (sometimes sends as strings)
# Fix: wrap ALL MCP tools, making every param a string. The wrapper
# parses JSON strings and coerces types back before calling the MCP tool.

def get_schema(tool):
    """Get JSON schema from a tool, handling both dict and Pydantic args_schema."""
    if isinstance(tool.args_schema, dict):
        return tool.args_schema
    if hasattr(tool.args_schema, 'model_json_schema'):
        return tool.args_schema.model_json_schema()
    return {}

def make_groq_wrapper(original_tool):
    """Wrap any MCP tool so ALL params are strings for Groq compatibility."""
    schema = get_schema(original_tool)
    props = schema.get("properties", {})
    required = set(schema.get("required", []))

    fields = {}
    for pname, pdef in props.items():
        desc = pdef.get("description", "")
        ptype = pdef.get("type", "string")

        # Add hint for object params
        if ptype == "object" or "properties" in pdef:
            default_val = pdef.get("default", {})
            desc += f' (Pass as JSON string, e.g. {json_mod.dumps(default_val)})'

        if pname in required:
            fields[pname] = (str, Field(description=desc))
        else:
            fields[pname] = (Optional[str], Field(default=pdef.get("default"), description=desc))

    ArgsModel = create_model(f"{original_tool.name}_Args", **fields)

    async def _call(**kwargs):
        fixed = {}
        for k, v in kwargs.items():
            if v is None:
                continue
            if isinstance(v, str):
                # Try to parse JSON strings back to objects/lists
                try:
                    parsed = json_mod.loads(v)
                    if isinstance(parsed, (dict, list)):
                        fixed[k] = parsed
                        continue
                    # Also recover integers/floats sent as strings
                    if isinstance(parsed, (int, float)):
                        fixed[k] = parsed
                        continue
                except (json_mod.JSONDecodeError, TypeError):
                    pass
            fixed[k] = v
        return await original_tool.ainvoke(fixed)

    return StructuredTool.from_function(
        coroutine=_call,
        name=original_tool.name,
        description=original_tool.description,
        args_schema=ArgsModel,
    )

# Wrap ALL MCP tools unconditionally for Groq compatibility
tools = []
for t in mcp_tools:
    wrapped = make_groq_wrapper(t)
    print(f"WRAPPED: {t.name}")
    tools.append(wrapped)

print(f"Total tools: {len(tools)}")

WRAPPED: search-flight
WRAPPED: feedback-to-devs
Total tools: 2


In [3]:
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: str, max_search_number: str) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters
      (e.g., use 'capacite' instead of 'capacité').
    """
    if int(search_number) > int(max_search_number):
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}


In [4]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [5]:
from langchain.agents import AgentState

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [6]:
from langchain.agents import create_agent

# Travel agent
travel_agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="""
    You are a travel agent. Search for flights to the desired destination wedding location.
    Today's date is 02/04/2026. All flight dates MUST be in the future (after today).
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    If the MCP tool fails, returns malformed output, or does not give you usable flight results, try the tool again.
    Once you have found the best options, let the user know your shortlist of options.
    """
)

In [7]:
# Venue agent

venue_agent = create_agent(
    model=model,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 12 web searches. Count every web_search call you make.
    After 12 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [8]:
# Playlist agent
playlist_agent = create_agent(
    model=model,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [9]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command
from groq import BadRequestError as GroqBadRequestError

def extract_failed_generation(exc):
    """Extract useful text from Groq's tool_use_failed errors."""
    try:
        body = exc.body
        if isinstance(body, dict) and "error" in body:
            fg = body["error"].get("failed_generation", "")
            if fg:
                return fg
    except Exception:
        pass
    return None

async def safe_subagent_invoke(agent, messages, is_async=True):
    """Invoke a subagent, catching Groq tool_use_failed errors and extracting the response."""
    try:
        if is_async:
            response = await agent.ainvoke({"messages": messages})
        else:
            response = agent.invoke({"messages": messages})
        return response['messages'][-1].content
    except ( GroqBadRequestError) as e:
        fg = extract_failed_generation(e)
        if fg:
            print(f"[Groq workaround] Extracted failed_generation ({len(fg)} chars)")
            return fg
        return f"Subagent error: {e}"
    except Exception as e:
        # Check if the nested cause has failed_generation
        cause = getattr(e, '__cause__', None) or getattr(e, '__context__', None)
        if cause:
            fg = extract_failed_generation(cause)
            if fg:
                print(f"[Groq workaround] Extracted failed_generation from cause ({len(fg)} chars)")
                return fg
        return f"Subagent error: {e}"

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    return await safe_subagent_invoke(
        travel_agent,
        [HumanMessage(content=f"Find flights from {origin} to {destination}")],
        is_async=True
    )

@tool
async def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    return await safe_subagent_invoke(
        venue_agent,
        [HumanMessage(content=f"Find wedding venues in {destination} for {capacity} guests")],
        is_async=False
    )

@tool
async def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    return await safe_subagent_invoke(
        playlist_agent,
        [HumanMessage(content=f"Find {genre} tracks for wedding playlist")],
        is_async=False
    )

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )

In [10]:
from langchain.agents import create_agent

coordinator = create_agent(
    model=model,
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for flights, venues, and playlists.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


## Test


In [11]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

/Users/chiranjeev/Documents/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='departureDateFlexRange', input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='returnDateFlexRange', input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(


[MCP interceptor] McpError on search-flight (code -32602, attempt 1/3): MCP error -32602: MCP error -32602: Invalid arguments for tool search-flight: [
  {
    "code": "custom",
    "message": "Dates must be in the future. Current date is 02/04/2026",
    "path": [
      "departureDate"
    ]
  }
]


/Users/chiranjeev/Documents/lca-lc-foundations/.venv/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='departureDateFlexRange', input_value=0, input_type=int])
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='returnDateFlexRange', input_value=0, input_type=int])
  return self.__pydantic_serializer__.to_python(


In [12]:
from pprint import pprint

pprint(response)

{'destination': 'Paris',
 'genre': 'jazz',
 'guest_count': '100',
 'messages': [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre", additional_kwargs={}, response_metadata={}, id='db5c708b-3e0e-4efc-ab29-927180c7c09e'),
              AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '09kyj5sx7', 'function': {'arguments': '{"destination":"Paris","genre":"jazz","guest_count":"100","origin":"London"}', 'name': 'update_state'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 959, 'total_tokens': 1011, 'completion_time': 0.133895159, 'completion_tokens_details': None, 'prompt_time': 0.02779558, 'prompt_tokens_details': None, 'queue_time': 0.4536543, 'total_time': 0.161690739}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_79da0e0073', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='

In [13]:
print(response["messages"][-1].content)

Based on the search results, I recommend booking a flight from London to Paris on April 3, 2026, at 19:55 with a layover in AMS, priced at £172 with a total duration of 11h 35m. 

For the wedding venue, I suggest considering Hotel Marignan Champs-Élysées, which can accommodate up to 100 guests for cocktails and aperitifs, or 80 for a sit-down meal, with a price range of €20,000-€30,000 for a classic wedding.

The curated playlist for your jazz-genre wedding includes 14 tracks with a total duration of 1 hour 45 minutes and 22 seconds, at a cost of $13.86.

Please let me know if you need any further assistance with coordinating your wedding.


link to trace: https://smith.langchain.com/public/7b5fe668-d3e3-4af4-b513-a8cacc0c9e84/r